In [ ]:
!pip install transformers onnxruntime onnx psutil matplotlib optimum optimum-onnx
!pip install onnxruntime-gpu==1.22.0

In [ ]:
import time, os
import torch
import pandas as pd
from transformers import AutoTokenizer, pipeline, DistilBertForSequenceClassification
import onnxruntime as ort # Added import

device = "cuda:0" if torch.cuda.is_available() else "cpu"
# Changed inputs to a list for batching
inputs = [f"This is sentence number {i+1} for batch size 128 testing." for i in range(128)]

if "CUDAExecutionProvider" in ort.get_available_providers() and device == "cuda:0":
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
else:
    providers = ["CPUExecutionProvider"]

def benchmark_inference(pipeline, inputs, n_runs=2):
    with torch.no_grad():
        start = time.perf_counter()
        for _ in range(n_runs):
            # Pass the list of inputs to the pipeline
            _ = pipeline(inputs)
        end = time.perf_counter()
        avg = (end - start) / (n_runs * len(inputs)) * 1000  # 毫秒
        return avg

def model_size(path):
    return os.path.getsize(path) / 1024 / 1024

# FP32 (baseline)





In [16]:
from transformers import AutoTokenizer, pipeline, BertForSequenceClassification # Changed import
import torch

def bert_fp32(model_checkpoint, device):

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    model = BertForSequenceClassification.from_pretrained( # Changed class
        model_checkpoint)
    model.to(device)

    # Create the pipeline, specifying the device
    return pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)

model_checkpoint = "XiaojingEllen/bert-finetuned-claim-detection"
text = "Hello, my dog is cute"
classifier = bert_fp32(model_checkpoint, device)
print(classifier(text))

baseline_ms = benchmark_inference(classifier, inputs)
print(baseline_ms)

Using device: cuda:0


Device set to use cuda:0


[{'label': 'LABEL_0', 'score': 0.990416407585144}]


# ONNX Runtime CPU - FP32



In [4]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer, pipeline
import torch

def bert_onnx_fp32(model_checkpoint, save_directory, providers, device):

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    ort_model = ORTModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        export=True,
        providers=providers)

    ort_model.save_pretrained(save_directory)
    tokenizer.save_pretrained(save_directory)
    return pipeline("text-classification", model=ort_model, tokenizer=tokenizer, device=device)

model_checkpoint = "XiaojingEllen/bert-finetuned-claim-detection"
text = "Hello, my dog is cute"
dir_bert = "onnx/bert"
ort_classifier = bert_onnx_fp32(model_checkpoint, dir_bert, providers, device)
print(ort_classifier(text))

ort_fp32_ms = benchmark_inference(ort_classifier, inputs)
print(ort_fp32_ms)

Multiple distributions found for package optimum. Picked distribution: optimum
The model distilbert/distilbert-base-uncased-finetuned-sst-2-english was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask
Device set to use cuda:0


[{'label': 'POSITIVE', 'score': 0.9997830986976624}]

# ONNX Runtime CPU - Dynamic - INT8


In [5]:
from optimum.onnxruntime import ORTQuantizer, ORTModelForSequenceClassification, pipeline
from optimum.onnxruntime.configuration import AutoQuantizationConfig

def bert_quantize(model_checkpoint, save_directory, providers, device):
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    onnx_model = ORTModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        export=True,
        providers = providers)

    # Create quantizer from model
    quantizer = ORTQuantizer.from_pretrained(onnx_model)

    # Define the dynamic quantization strategy by creating the configuration
    dqconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)

    # Quantize the model
    model_quantized_path = quantizer.quantize(
        save_dir=save_directory,
        quantization_config=dqconfig
        )

    # inference
    model_dyn_u8s8 = ORTModelForSequenceClassification.from_pretrained(
        save_directory,
        file_name="model_quantized.onnx",
        providers = providers)

    return pipeline("text-classification", model=model_dyn_u8s8, tokenizer=tokenizer, device=device)


model_checkpoint = "XiaojingEllen/bert-finetuned-claim-detection"
dir_quantized = "onnx/bert-int8"
ort_int8_classifier = bert_quantize(model_checkpoint, dir_quantized, providers, device)
print(ort_int8_classifier(text))

ort_int8_ms = benchmark_inference(ort_int8_classifier, inputs)
print(ort_int8_ms)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

The model distilbert-base-uncased-finetuned-sst-2-english was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Device set to use cuda:0


[{'label': 'POSITIVE', 'score': 0.999778687953949}]


## Optimizing with ORTOptimizer

four common optimization levels:
* O1: basic general optimizations.
* O2: basic and extended general optimizations, transformers-specific fusions.
* O3: same as O2 with GELU approximation.
* O4: same as O3 with mixed precision (fp16, GPU-only).

In [ ]:
from optimum.onnxruntime import (
    ORTModelForSequenceClassification,
    ORTOptimizer
)
from optimum.onnxruntime.configuration import AutoOptimizationConfig
from optimum.onnxruntime import pipeline

def bert_optimize(model_checkpoint, save_directory, providers, device):

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    model = ORTModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        export=True,
        providers = providers)

    # Load the optimization configuration detailing the optimization we wish to apply
    optimization_config = AutoOptimizationConfig.O4()
    optimizer = ORTOptimizer.from_pretrained(model)

    optimizer.optimize(save_dir=save_directory, optimization_config=optimization_config)
    optimized_model = ORTModelForSequenceClassification.from_pretrained(
        save_directory,
        file_name="model_optimized.onnx",
        providers=providers)

    return pipeline("text-classification", model=optimized_model, tokenizer=tokenizer, device=device)

model_checkpoint = "XiaojingEllen/bert-finetuned-claim-detection"
dir_optimized = "onnx/bert-optimized"
text = "I like the new ORT pipeline"
ort_optimize_ms = bert_optimize(model_checkpoint, dir_optimized, providers, device)
print(ort_optimize_ms(text))

ort_optimize_ms = benchmark_inference(ort_optimize_ms, inputs)
print(ort_optimize_ms)

# ONNX Runtime GPU - FP32 - IOBinding

Due to current limitations in ONNX Runtime, it is not possible to use quantized models with CUDAExecutionProvider.

IOBinding is an efficient way to avoid expensive data copying when using GPUs.

These data copying overheads between the host and devices are expensive, and can lead to worse inference latency than vanilla PyTorch especially for the decoding process.

In [8]:
import onnxruntime
from transformers import AutoTokenizer, pipeline
from optimum.onnxruntime import ORTModelForSeq2SeqLM

session_options = onnxruntime.SessionOptions()
session_options.log_severity_level = 0

def translator_ort_fp32(model_checkpoint, save_dir, device):
      tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
      model_t5 = ORTModelForSeq2SeqLM.from_pretrained(
          model_checkpoint,
          export=True,
          use_io_binding=None,
          providers = ["CUDAExecutionProvider"])

      model_t5.save_pretrained(save_dir)
      tokenizer.save_pretrained(save_dir)

      return pipeline("translation_en_to_fr", model=model_t5, tokenizer=tokenizer, device=device)

model_checkpoint = "t5-small"
dir_t5 = "onnx/t5-small/"
onnx_translation = translator_ort_fp32(model_checkpoint, dir_t5, device)
print(onnx_translation(text))

gpu_ms = benchmark_inference(onnx_translation, inputs)
print(gpu_ms)

def translator_ort_fp32_io_binding(model_checkpoint, save_dir, device):
      tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
      model_t5_io_binding = ORTModelForSeq2SeqLM.from_pretrained(
          model_checkpoint,
          export=True,
          use_io_binding=True,
          providers = ["CUDAExecutionProvider"])

      model_t5_io_binding.save_pretrained(save_dir)
      tokenizer.save_pretrained(save_dir)

      return pipeline("translation_en_to_fr", model=model_t5_io_binding, tokenizer=tokenizer, device=device)

dir_t5_io = "onnx/t5-small-io-binding/"
onnx_translation_io_binding = translator_ort_fp32_io_binding(model_checkpoint, dir_t5_io, device)
print(onnx_translation_io_binding(text))

gpu_io_binding_ms = benchmark_inference(onnx_translation_io_binding, inputs)
print(gpu_io_binding_ms)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

The model t5-small was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/t5/modeling_t5.py:1263: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if sequence_length != 1:
/usr/local/lib/python3.12/dist-packages/transformers/cache_utils.py:108: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.keys is None or self.keys.numel() == 0:
Device set to use cuda:0


[{'translation_text': 'Bonjour, mon chien est cute'}]


The model t5-small was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`
Device set to use cuda:0


[{'translation_text': 'Bonjour, mon chien est cute'}]


# TensorRT - INT8


In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
version="10.13.0.35"
arch="x86_64"
cuda="cuda-12.9"
filename = f"TensorRT-{version}.Linux.{arch}-gnu.{cuda}.tar.gz"
!tar -xzvf {filename}

TensorRT-10.13.0.35/
TensorRT-10.13.0.35/bin
TensorRT-10.13.0.35/include/
TensorRT-10.13.0.35/include/NvInferPluginUtils.h
TensorRT-10.13.0.35/include/NvInferLegacyDims.h
TensorRT-10.13.0.35/include/NvInferVersion.h
TensorRT-10.13.0.35/include/NvInfer.h
TensorRT-10.13.0.35/include/NvInferRuntimePlugin.h
TensorRT-10.13.0.35/include/NvOnnxConfig.h
TensorRT-10.13.0.35/include/NvInferPlugin.h
TensorRT-10.13.0.35/include/NvInferRuntime.h
TensorRT-10.13.0.35/include/NvInferRuntimeCommon.h
TensorRT-10.13.0.35/include/NvInferImpl.h
TensorRT-10.13.0.35/include/NvOnnxParser.h
TensorRT-10.13.0.35/include/NvInferRuntimeBase.h
TensorRT-10.13.0.35/include/NvInferPluginBase.h
TensorRT-10.13.0.35/samples/
TensorRT-10.13.0.35/samples/sampleOnnxMnistCoordConvAC/
TensorRT-10.13.0.35/samples/sampleOnnxMnistCoordConvAC/modify_onnx_ac.py
TensorRT-10.13.0.35/samples/sampleOnnxMnistCoordConvAC/sampleOnnxMnistCoordConvAC.cpp
TensorRT-10.13.0.35/samples/sampleOnnxMnistCoordConvAC/Makefile
TensorRT-10.13.0.35/sa

In [12]:
!export LD_LIBRARY_PATH=TensorRT-${version}/lib:$LD_LIBRARY_PATH

In [8]:
%env LD_LIBRARY_PATH=/content/TensorRT-10.13.0.35/lib:$LD_LIBRARY_PATH

env: LD_LIBRARY_PATH=/content/TensorRT-10.13.0.35/lib:$LD_LIBRARY_PATH


In [2]:
!python --version

Python 3.12.12


In [11]:
!python3 -m pip install TensorRT-10.13.0.35/python/tensorrt-*-cp312-none-linux_x86_64.whl --force-reinstall

Processing ./TensorRT-10.13.0.35/python/tensorrt-10.13.0.35-cp312-none-linux_x86_64.whl
  Attempting uninstall: tensorrt
    Found existing installation: tensorrt 10.13.0.35
    Uninstalling tensorrt-10.13.0.35:
      Successfully uninstalled tensorrt-10.13.0.35


In [22]:
import onnxruntime as ort
print(ort.get_available_providers())

['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [15]:
import onnxruntime
from optimum.onnxruntime import ORTModelForCausalLM, AutoQuantizationConfig
import os

os.makedirs("tmp/trt_cache_gpt2", exist_ok=True)

session_options = onnxruntime.SessionOptions()
session_options.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_DISABLE_ALL

tokenizer = AutoTokenizer.from_pretrained("fxmarty/distilbert-base-uncased-sst2-onnx-int8-for-tensorrt")

tensor_model = ORTModelForSequenceClassification.from_pretrained(
    "fxmarty/distilbert-base-uncased-sst2-onnx-int8-for-tensorrt",
    provider="TensorrtExecutionProvider",
    session_options=session_options,
    provider_options={"trt_int8_enable": True},
)

#qconfig = AutoQuantizationConfig.tensorrt(per_channel=False)
text = ["TensorRT is a bit painful to use, but at the end of day it runs smoothly and blazingly fast!"]
inp = tokenizer(text, return_tensors="np").to("cuda")

res = tensor_model(**inp)

print(res)
print(tensor_model.config.id2label[res.logits[0].argmax()])


tokenizer_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/709 [00:00<?, ?B/s]

Could not find any ONNX files with standard file name model.onnx, files found: [PosixPath('model_quantized.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.


./model_quantized.onnx:   0%|          | 0.00/268M [00:00<?, ?B/s]

*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/python/onnxruntime_pybind_state.cc:505 void onnxruntime::python::RegisterTensorRTPluginsAsCustomOps(PySessionOptions&, const onnxruntime::ProviderOptions&) Please install TensorRT libraries as mentioned in the GPU requirements page, make sure they're in the PATH or LD_LIBRARY_PATH, and that your GPU is supported.
 when using ['TensorrtExecutionProvider']
Falling back to ['CPUExecutionProvider'] and retrying.
****************************************
SequenceClassifierOutput(loss=None, logits=array([[-0.72605634,  0.734425  ]], dtype=float32), hidden_states=None, attentions=None)
POSITIVE


In [18]:
# display
data = {
    '模型': ['baseline bert FP32', 'bert ort FP32', 'bert ort INT8', 'bert ort mixed', 't5-small FP32', 't5-small FP32'],
    '设备': ['GPU', 'GPU', 'GPU', 'GPU', 'GPU', 'GPU'],
    'IO Binding': ["Y", "Y", "Y", "Y", "N", "Y"],
    '延迟(ms)': [baseline_ms, ort_fp32_ms, ort_int8_ms, ort_optimize_ms, gpu_ms, gpu_io_binding_ms],
    'onnx_size(MB)': ['-', model_size(f'{dir_bert}/model.onnx'),
                      model_size(f'{dir_quantized}/model_quantized.onnx'),
                      model_size(f'{dir_optimized}/model_optimized.onnx'),
                      model_size(f'{dir_t5}/encoder_model.onnx') + model_size(f'{dir_t5}/decoder_model.onnx'),
                      model_size(f'{dir_t5_io}/encoder_model.onnx')+ model_size(f'{dir_t5_io}/decoder_model.onnx')]
}

df = pd.DataFrame(data)

# Format the '延迟(ms)' column
df['延迟(ms)'] = df['延迟(ms)'].map('{:.2f}'.format)
df['onnx_size(MB)'] = df['onnx_size(MB)'].apply(
    lambda x: '{:.2f}'.format(x) if isinstance(x, (int, float)) and not pd.isna(x) else x
)

display(df)

Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask
Device set to use cuda:0
Device set to use cuda:0
The model t5-small was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`
/usr/local/lib/python3.12/dist-packages/transformers/models/t5/modeling_t5.py:1263: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future.

,模型,设备,IO Binding,延迟(ms),onnx_size(MB)
0,bert baseline FP32,GPU,Y,8.10,0.00
1,bert ort FP32,GPU,Y,4.26,417.85
2,bert ort INT8,GPU,Y,18.57,105.17
3,t5-small FP32,GPU,N,131.59,356.58
4,t5-small FP32,GPU,Y,130.56,356.58
